In [1]:
from model_engine.assets.utils import load_asset, get_default_factory_configs

from model_engine.model_builder.asset_parser import asset_parser
from model_engine.client_predictor.client_predictor import ClientPredictor
from model_engine.assets.power.app_asset import schema
from model_engine.model_builder import ModelBuilder
from model_engine.io.utils import global_load_data, global_archive_data, global_path_exists
from model_engine.power.post_sale.base.utils import normalize_io
from model_engine.power.post_sale.base import ClientHandler, ModelConstraints, PowerModelingBase, S3ClientHandler
from model_engine.power.post_sale import ClientModelBuilder
from model_engine.io.loaders import DictHelper
from model_engine.io.s3_schema_manager import S3SchemaManager
import s3fs
import pandas as pd
from zestio import DataArchiver
import copy
from projectz import ZInfo
from zestio import handler
from model_engine.io.loaders import data_loader
from zestio.handler import FileSystemHandler, S3Handler
import json
import os
from zestio import handler
from zaml.common.utils.io import load_state
from zestio import handler

/home/jag/.local/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



In [2]:
def connect_client(z_info, client_name):
    z_info.connect_client(name = client_name)
    return z_info
def connect_datasets(z_info, dataset_version):
    datasets = z_info.context.web_handler.get(f"clients/{z_info.client.id}/datasets")
    dataset_num = dataset_version-1
    dataset_model_type = datasets[dataset_num]['model_type']
    dataset_data_source = datasets[dataset_num]['data_source']
    dataset_product = datasets[dataset_num]['product']
    dataset_version = datasets[dataset_num]['version']
    z_info.connect_dataset(model_type = dataset_model_type,
                                     product = dataset_product, 
                                     version = dataset_version)
    return z_info
def connect_model(z_info, model_name, product):
    z_info.connect_model_iteration(name = model_name, product = product,  model_type=['underwriting'])
    return z_info
def connect_to_z_info(dataset_version, product, model_name, client_name):
    z_info = ZInfo()
    z_info =connect_client(z_info, client_name)
    z_info = connect_datasets(z_info, dataset_version)
    z_info = connect_model(z_info, model_name, product)
    return z_info
def get_alternate_paths(processed_path):
    """Takes a PROCESSED path and returns PREPROCESSED and NORMALIZED paths"""
    # PROCESSED:    TABLE=X/VERSION/CLIENT/PRODUCT/BUREAU/FORMAT/PULL_DATE/PULL_NAME/ME_VERSION
    # PREPROCESSED: BUREAU/FORMAT/TABLE=X/VERSION/CLIENT/PRODUCT/PULL_DATE/PULL_NAME/ME_VERSION  
    # NORMALIZED:   TABLE=X/VERSION/CLIENT/PRODUCT/BUREAU/FORMAT (no PULL_DATE onwards? - check screenshot)
    
    # Extract partitions from processed path
    parts = processed_path.replace('s3://power-client-data-staging/PROCESSED/DATA/', '').rstrip('/').split('/')
    partitions = {}
    for p in parts:
        if '=' in p:
            key, val = p.split('=', 1)
            partitions[key] = val
    
    base = 's3://power-client-data-staging'
    
    preprocessed = (f"{base}/PREPROCESSED/DATA/"
                    f"BUREAU={partitions['BUREAU']}/FORMAT={partitions['FORMAT']}/"
                    f"TABLE={partitions['TABLE']}/VERSION={partitions['VERSION']}/"
                    f"CLIENT={partitions['CLIENT']}/PRODUCT={partitions['PRODUCT']}/"
                    f"PULL_DATE={partitions['PULL_DATE']}/PULL_NAME={partitions['PULL_NAME']}/"
                    f"ME_VERSION={partitions['ME_VERSION']}/")
    
    normalized = (f"{base}/NORMALIZED/DATA/"
                  f"TABLE={partitions['TABLE']}/VERSION={partitions['VERSION']}/"
                  f"CLIENT={partitions['CLIENT']}/PRODUCT={partitions['PRODUCT']}/"
                  f"BUREAU={partitions['BUREAU']}/FORMAT={partitions['FORMAT']}/")
    
    return preprocessed, normalized

In [3]:
dataset_version = 1
product = ['autoloan']
model_name = 'model1_AppData_LTVAutoloan'
client_name = 'californiacu'

z_info = connect_to_z_info(dataset_version, product, model_name, client_name)

INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected to client: californiacu
INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected to dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected to model_iteration: autoloan/model1_AppData_LTVAutoloan


In [4]:
model_artifacts_dir = z_info.model_iteration._dir_paths['modeling_model_artifacts_dir']

In [9]:
model_artifacts_dir

'/d/shared/silver_projects_v2/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/model_artifacts/autoloan/model1_AppData_LTVAutoloan'

# The Asset contains the paths to our directory. It is created in the final clientpredictor and is the model builder asset

In [57]:
asset = load_asset(os.path.join(model_artifacts_dir, 'asset.json'))

In [58]:
asset['config']

{'bivariate_fe_instructions': [{'feature_1': 'client_data_AmountRequested',
   'feature_2': 'client_data_TotalVehicleValue',
   'operation': 'divide',
   'new_feature': 'biv_loan_to_value'}],
 'monotonic_constraints_list': [{'feature': 'biv_loan_to_value',
   'monotonic_constraint': 'positive'}],
 'exclusion_list': [{'feature': 'client_data_AmountRequested',
   'reason': 'used indirectly'},
  {'feature': 'client_data_TotalVehicleValue', 'reason': 'used indirectly'},
  {'feature': 'TotalVehicleValue', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncomeBaseSalary', 'reason': 'Not Used'},
  {'feature': 'AmountRequested', 'reason': 'Used Indirectly'},
  {'feature': 'MonthlyIncome', 'reason': 'Not Used'},
  {'feature': 'VehicleValue', 'reason': 'Not Used'},
  {'feature': 'LoanTerm', 'reason': 'Not Used'},
  {'feature': 'LTV', 'reason': 'Not Used'},
  {'feature': 'DTI', 'reason': 'Not Used'},
  {'feature': 'PTI', 'reason': 'Not Used'},
  {'feature': 'payment_to_income', 'reason': 'Not

In [59]:
asset['data'].keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target', 'config', 'client_data'])

# We can see that we can see the location of the assets we used in model-engine

* The InputNormalizer Asset is [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/equifax/cms_6/fe2/trade.json)
* The AggregationEngine Asset is [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/aggregation/fe2/trade.json)

In [60]:
asset['data']['trade']

{'asset': {'fe_version': 2,
  'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
  'AggregationEngine': 'aggregation/fe2/trade.json'},
 'data': ['s3://power-client-data-staging/PROCESSED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'],
 'io_params': {'drop_duplicates': True,
  'keep_features': ['trade_count__all_accounts',
   'trade_months_since_oldest_account_opened__all_accounts',
   'trade_months_since_most_recent_DQ30_or_greater__all_accounts',
   'trade_months_since_most_recent_DQ60_or_greater__all_accounts',
   'trade_percent_never_dq__all_accounts',
   'trade_percent_derog__all_accounts',
   'trade_percent_accounts_with_DQ30_or_greater_in_last_24_months__all_accounts',
   'trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__all_accounts',
   'trade_percent_accounts_with_DQ120_or_great

# Note that for this one we have a member asset because our KOB is a credit-union

# Also note our data here points to the processed after normalization and aggregation: we don't want to re-run the feature engineering when we train the model: but the assets and transformer could work with it being preprocessed

In [61]:
asset['data'].keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target', 'config', 'client_data'])

# Note on app and config: Same data source except one is for creating-application level features and one for that client: product, region, KOB 

In [62]:
asset['data']['app']

{'asset': {'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'categoricals': {'is_client': {'True': 'True', 'False': 'False'}}},
  'table_name': 'app',
  'table_type': 'one_to_one',
  'feature_engineering': 'one_to_one'},
 'data': ['s3://power-client-data-staging/PROCESSED/DATA/TABLE=analysis_data/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'],
 'io_params': {'drop_duplicates': True,
  'keep_features': ['appDate',
   'is_client',
   'bisgWhite',
   'bisgBlack',
   'bisgAian',
   'bisgApi',
   'bisgHispanic',
   'bisgMulti',
   'prob_male',
   'prob_female',
   'prob_<62',
   'prob_62+',
   'VS30',
   'VS40',
   'perf_state_cde',
   'perf_acct_typ_cde',
   'kob',
   'flg_exist_fe_trade',
   'ZEST_KEY'],
  'memory_efficient': False}}

# We can see the first part in the placeholders is the model_settings: tells us which of the binary variables created below we will actually turn on based on the info for this client

In [63]:
asset['data']['config']

{'data': ['s3://power-client-data-staging/PROCESSED/DATA/TABLE=analysis_data/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'],
 'io_params': {'drop_duplicates': True, 'memory_efficient': False},
 'asset': {'feature_engineering': 'one_to_one',
  'validator': 'base',
  'table_name': 'config',
  'table_type': 'one_to_one',
  'info': {'key': 'ZEST_KEY',
   'app_date': 'appDate',
   'categoricals': {},
   'constant_categorical_feature': {'product': 'autoloan',
    'region': 'west',
    'KOB': 'creditunion'},
   'placeholders': {'encoding_type': 'one-hot',
    'drop_original_features': True,
    'placeholders': {'KOB': {'creditunion': {'flag': True,
       'val': 'creditunion'},
      'regionalbank': {'flag': True, 'val': 'regionalbank'}},
     'product': {'autoloan': {'flag': True, 'val': 'autoloan'},
      'creditcard': {'f

In [64]:
os.listdir(os.path.join(model_artifacts_dir,'fe_data')), os.listdir(os.path.join(model_artifacts_dir, 'fe_data', 'split=test'))

(['split=test', 'split=train'], ['data=client'])

In [65]:
fe_data = pd.read_parquet(os.path.join(model_artifacts_dir, 'fe_data', 'split=test','data=client'))

In [66]:
trade_columns = ['ZEST_KEY'] + [col for col in fe_data.columns if col.startswith('trade')] 

In [67]:
fe_data[trade_columns]

,ZEST_KEY,trade_max_balance_to_hc__with_historic_utilization_over_50_percent_revolving,trade_count__non_derog_closed_accounts,trade_max_credit_limit__with_utilization_over_25_percent_open_cc,trade_count__with_recent_payment_open_accounts,trade_count__non_derog_accounts,trade_mean_historic_utilization__with_recent_payment_open_cc,trade_percent_derog__all_accounts,trade_percent_accounts_with_DQ30_or_greater_in_last_6_months__with_recent_payment_open_accounts,trade_mean_percent_unpaid__with_recent_payment_personal,...,trade_mean_balance_to_hc__active_open_revolving,trade_mean_historic_utilization__non_derog_open_cc,trade_sum_loan_amount__never_delinquent_closed_installment,trade_sum_balance__unsecure_open_accounts,trade_max_historic_utilization__with_utilization_over_50_percent_revolving,trade_percent_recently_open__with_historic_utilization_over_100_percent_cc,trade_months_since_oldest_account_opened__secure_accounts,trade_count__non_derog_closed_unsecure_personal,trade_max_utilization__non_derog_open_revolving,trade_max_past_due_to_monthly_payment__all_closed_auto
0,303683_1_276_12,0.687027,0.0,-3.0,2.0,4.0,-3.000000,0.428571,0.0,-3.000000,...,0.843513,0.088000,-2.0,2900.0,0.909556,-3.0,25.856794,0.0,0.624889,-2.0
1,303687_1_276_12,-2.000000,0.0,-2.0,0.0,0.0,-2.000000,1.000000,-2.0,-2.000000,...,-2.000000,-2.000000,-2.0,-2.0,-2.000000,-2.0,-3.000000,0.0,-2.000000,-2.0
2,303691_1_276_12,0.887457,6.0,10000.0,5.0,13.0,0.923969,0.000000,0.0,0.744103,...,0.544727,0.923969,6615.0,11847.0,0.956692,-3.0,107.566890,1.0,0.849023,-2.0
3,303705_2_276_12,0.569787,14.0,15000.0,5.0,20.0,0.807408,0.000000,0.0,0.872617,...,0.519402,0.807408,-2.0,5395.0,-3.000000,0.0,234.616727,0.0,0.482500,-2.0
4,303705_1_276_12,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.000000,-1.0,-1.000000,...,-1.000000,-1.000000,-1.0,-1.0,-1.000000,-1.0,-1.000000,-1.0,-1.000000,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12288,344653_2_276_22,-3.000000,0.0,-3.0,1.0,2.0,0.243000,0.000000,0.0,-2.000000,...,0.010288,0.243000,-2.0,4760.0,-3.000000,-3.0,-3.000000,0.0,0.005000,-2.0
12289,344654_2_276_22,0.000000,24.0,-3.0,3.0,31.0,0.245000,0.000000,0.0,-2.000000,...,0.028734,0.155121,8580.0,721.0,-3.000000,0.0,69.093821,0.0,0.048067,-4.0
12290,344654_1_276_22,0.000000,8.0,-2.0,1.0,10.0,-2.000000,0.000000,0.0,-2.000000,...,-2.000000,-2.000000,39303.0,-3.0,-3.000000,-3.0,149.883981,0.0,-2.000000,-4.0
12291,344655_1_276_22,0.758442,0.0,2500.0,1.0,2.0,0.924000,0.000000,0.0,-2.000000,...,0.379221,0.561250,-2.0,1752.0,0.924000,-3.0,-3.000000,0.0,0.700800,-2.0


In [68]:
asset['data']['trade']['data']

['s3://power-client-data-staging/PROCESSED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/']

In [69]:
from model_engine.io.loaders import data_loader
trade_data = data_loader(asset['data']['trade']['data'], asset={'info': {'key': 'ZEST_KEY'}}).reset_index()[trade_columns]

In [70]:
[feature for feature in trade_data.columns if feature not in asset['data']['trade']['io_params']['keep_features']]

['ZEST_KEY']

# Note, that these keep_features will map the one in our national model: more on that in the other file

# We can see that the data is the same for people in our FE who have tradelines

In [71]:
merged_trade_data = trade_data.merge(fe_data[['ZEST_KEY']], on='ZEST_KEY', how='inner')
merged_fe_data_trade_columns = fe_data[trade_columns].merge(trade_data[['ZEST_KEY']], on = 'ZEST_KEY', how = 'inner')

# For people who have a tradeline it is the same as the processed data

In [72]:
import numpy as np
np.unique(merged_trade_data == merged_fe_data_trade_columns)

array([ True])

# We can find the original preprocessed data

In [28]:
processed_trade = asset['data']['trade']['data'][0]

In [29]:
asset['data'].keys()

dict_keys(['trade', 'inq', 'collec', 'bnkr', 'member', 'app', 'target', 'config', 'client_data'])

In [30]:
processed_inquiry = asset['data']['inq']['data'][0]

In [31]:
preprocessed_path, normalized_path = get_alternate_paths(processed_trade)

In [32]:
preprocessed_path_inq, normalized_path_inq = get_alternate_paths(processed_inquiry)

# Let's subset preprocessed to only have 500 of our members

# We can confirm no data leakage: all of our dates are before the application date
# Likewise see the archive data

In [33]:
members_to_select = fe_data.reset_index()[['ZEST_KEY']][0:500]

In [44]:
preprocessed_path

's3://power-client-data-staging/PREPROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'

In [45]:
asset['data']['trade']['data'][0]

's3://power-client-data-staging/PROCESSED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'

In [46]:
normalized_path

's3://power-client-data-staging/NORMALIZED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/'

In [47]:
all_preprocessed_inq = pd.read_parquet(preprocessed_path_inq)

In [48]:
# distinct number of quarter values per zest_key
quarter_counts_inq = (
    all_preprocessed_inq.groupby("ZEST_KEY")["quarter"]
    .nunique()
    .sort_values(ascending=False)
)

# max distinct quarter count across zest_keys
max_count_inq = quarter_counts_inq.max()

print("Max distinct quarter count inq:", max_count_inq)



Max distinct quarter count inq: 1


In [49]:
all_preprocessed = pd.read_parquet(preprocessed_path)

In [50]:
all_preprocessed.columns

Index(['ZEST_KEY', 'quarter', 'appDate', 'SEG_SEQ', 'SEG_PARENT',
       'SEG_PARENT_SEQ', 'SEGMENT_TYPE', 'CUSTOMER_NAME', 'CUSTOMER_NUMBER',
       'DATE_REPORTED', 'DATE_OPENED', 'HIGH_CREDIT', 'CREDIT_LIMIT',
       'BALANCE', 'PAST_DUE_AMOUNT', 'PORTFOLIO_TYPE', 'RATE_STATUS_CODE',
       'AUTOMATED_UPDATE_INDICATOR', 'MONTHS_REVIEWED', 'ECOA_DESIGNATOR',
       'ACCOUNT_NUMBER', '30_DAY_COUNTERS', '60_DAY_COUNTERS',
       '90+_DAY_COUNTERS', 'PREVIOUS_HIGH_RATE_1', 'PREVIOUS_HIGH_DATE_1',
       'PREVIOUS_HIGH_RATE_2', 'PREVIOUS_HIGH_DATE_2', 'PREVIOUS_HIGH_RATE_3',
       'PREVIOUS_HIGH_DATE_3', 'DFD_DLA', 'NARRATIVE_CODE_1',
       'NARRATIVE_CODE_2', 'NARRATIVE_CODE_3', 'NARRATIVE_CODE_4',
       'ACCOUNT_TYPE', 'LAST_PAYMENT_DATE', 'CLOSED_DATE', 'DMD_REPORTED',
       'ACTUAL_PAYMENT_AMOUNT', 'SCHEDULED_PAYMENT_AMOUNT', 'TERMS_FREQUENCY',
       'TERMS_DURATION', 'INDICATOR', 'NAME', 'CREDITOR_CLASSIFICATION',
       'ACTIVITY_DESIGNATOR', 'ORIGINAL_CHARGE_OFF_AMOUNT',
    

In [51]:
# distinct number of quarter values per zest_key
quarter_counts = (
    all_preprocessed.groupby("ZEST_KEY")["quarter"]
    .nunique()
    .sort_values(ascending=False)
)

# max distinct quarter count across zest_keys
max_count = quarter_counts.max()

print("Max distinct quarter count:", max_count)

# optional: see all zest_keys tied for the max
quarter_counts[quarter_counts == max_count]

Max distinct quarter count: 1


ZEST_KEY
210623_1_276_12    1
388237_2_276_12    1
388307_1_276_12    1
388305_1_276_22    1
388300_1_276_12    1
                  ..
274557_1_276_12    1
274554_1_276_12    1
274551_1_276_22    1
274549_1_276_22    1
531926_2_276_12    1
Name: quarter, Length: 99437, dtype: int64

In [52]:
all_preprocessed[['quarter', 'appDate']].dtypes

quarter     period[Q-DEC]
appDate    datetime64[ns]
dtype: object

In [53]:
preprocessed = all_preprocessed.merge(members_to_select, on = 'ZEST_KEY', how = 'inner')

In [54]:
preprocessed_inq = all_preprocessed_inq.merge(members_to_select, on = 'ZEST_KEY', how = 'inner')

In [55]:
members_to_select

,ZEST_KEY
0,303683_1_276_12
1,303687_1_276_12
2,303691_1_276_12
3,303705_2_276_12
4,303705_1_276_12
...,...
495,305370_1_276_12
496,305371_1_276_12
497,305375_1_276_12
498,305378_1_276_12


In [56]:
all_preprocessed

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,PREVIOUS_HIGH_DATE_BEFORE_HISTORY,NARRATIVE_CODE_1X,NARRATIVE_CODE_2X,NARRATIVE_CODE_3X,NARRATIVE_CODE_4X,RECENT_TRADE_FLAG,QUALIFYING_FLAG,ARCHIVE_DATA,DATE_OF_REQUEST,ARCHIVE_DATE
0,210623_1_276_12,2021Q2,2021-07-01,1,FULL-Header,1,PT,None,BB,06232021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
1,210623_1_276_12,2021Q2,2021-07-01,2,FULL-Header,2,PT,None,BB,06162021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
2,210623_1_276_12,2021Q2,2021-07-01,3,FULL-Header,3,PT,None,OC,05212021,...,None,166,None,None,None,Y,N,None,2021-06-29,2021-06-30
3,210623_1_276_12,2021Q2,2021-07-01,4,FULL-Header,4,PT,None,ON,05172021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
4,210623_1_276_12,2021Q2,2021-07-01,5,FULL-Header,5,PT,None,ON,04212021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1845433,531912_2_276_22,2025Q3,2025-12-31,10,FULL-Header,10,PT,None,FM,11062020,...,None,208,132,None,None,Y,Y,None,2025-09-30,2025-09-30
1845434,531912_2_276_22,2025Q3,2025-12-31,11,FULL-Header,11,PT,None,CG,07282018,...,None,065,None,None,None,Y,Y,None,2025-09-30,2025-09-30
1845435,531912_2_276_22,2025Q3,2025-12-31,12,FULL-Header,12,PT,None,FA,01312018,...,None,None,None,None,None,Y,Y,None,2025-09-30,2025-09-30
1845436,531912_2_276_22,2025Q3,2025-12-31,13,FULL-Header,13,PT,None,BB,12312017,...,None,None,None,None,None,Y,Y,None,2025-09-30,2025-09-30


In [ ]:
# Get the end timestamp of the quarter period
quarter_end_dates = all_preprocessed['quarter'].dt.end_time


In [ ]:
quarter_end_dates_inq = all_preprocessed_inq['quarter'].dt.end_time

is_before_inq = quarter_end_dates_inq < all_preprocessed_inq['appDate']

# Check if this condition is True for ALL rows in the dataframe
all_quarters_before_inq= is_before_inq.all()

print(f"Does the quarter always end before the appDate? {all_quarters_before_inq}")


if not all_quarters_before:
    print("\nHere are the rows where the quarter does NOT occur before the appDate:")
    violations = all_quarters_before_inq[~is_before_inq]
    display(violations[['quarter', 'appDate']])

In [ ]:


is_before =  < all_preprocessed['appDate']

# Check if this condition is True for ALL rows in the dataframe
all_quarters_before = is_before.all()

print(f"Does the quarter always end before the appDate? {all_quarters_before}")


if not all_quarters_before:
    print("\nHere are the rows where the quarter does NOT occur before the appDate:")
    violations = all_preprocessed[~is_before]
    display(violations[['quarter', 'appDate']])

In [ ]:
all_preprocessed['DATE_REPORTED'].dtype

# How did we convert from preprocessed to normalized?

In [ ]:
asset['data']['trade']['asset']

# We can see that for our InputNormalizer we used the asset `equifax/cms_6/fe2/trade.json`

* The InputNormalizer asset contains the mapping and preprocessing instructions for taking raw bureau data (columns like `BALANCE`, `DATE_OPENED`, `CREDIT_LIMIT`) and converting them to canonical column names and intermediate features that the AggregationEngine expects
* The goal of normalization is to get different bureaus (Equifax, Experian, Transunion) into the same standardized format - so we can apply one AggregationEngine to all of them regardless of which bureau the data came from
* This is what runs in production when the data arrives only preprocessed - in our training case the data is already processed so the FE engine passes through, but the asset is still included in the pipeline so it works in both contexts

## Thus, if the asset were a map, the InputNormalizer is the set of directions for how to get from raw bureau columns to a standardized format

In [ ]:
asset_trade_normalizer = load_asset(asset['data']['trade']['asset']['InputNormalizer'])

In [ ]:
pipeline = load_state(os.path.join(model_artifacts_dir, 'pipeline.obj'))

In [ ]:
node_dict = {}
for node in pipeline.graph.nodes:
    print(node.name)
    node_dict[node.name] = node
    

# We can see that our trade feature transformer is an EndToEndFeatureEngine

In [ ]:
node_dict['trade FE'].transformer

# Furthermore,  we can see that it again points to the same parts of the asset: the normalizer and the aggregation engine

In [ ]:
node_dict['trade FE'].transformer.asset.keys()

## We can see that the InputNormalizer Asset here is the same as the asset we had before

In [ ]:
node_dict['trade FE'].transformer.asset['InputNormalizer']

In [ ]:
node_dict['trade FE'].transformer.asset['InputNormalizer'] == asset_trade_normalizer

# We can see inside the transformer it has two transformer: an InputNormalizer and an Aggregation Engine

In [ ]:
node_dict['trade FE'].transformer.transformers

In [ ]:
node_dict['trade FE'].transformer.transformers['InputNormalizer']

## Thus, if the InputNormalizer asset is the set of directions, then the `InputNormalizer` object is the driver that actually follows those directions - it reads the asset's mapping and preprocessing instructions and executes the transformations on the data

# We can see the source code for the InputNormalizer object [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/feature_engine_V2/feature_engine.py#L149)

* It has two main operations: **mapping** (converts raw bureau columns to canonical names and types - e.g., `BALANCE` -> `balance_amt` as numeric, `DATE_OPENED` -> `openDate` as date, `TERMS_FREQUENCY` -> `termFreqMult` with a frequency-to-multiplier mapping) and **preprocessing** (creates derived intermediate columns and binary filter flags from the mapped data - e.g., `months_since_openDate` from date diffs, `is_open`/`is_closed`/`is_revolving`/`is_auto` from account codes and narrative codes, `crdUtl` from balance/credit_limit ratios, and delinquency status flags from payment history patterns)
* The goal of normalization is to get different bureaus (Equifax, Experian, Transunion) into the same standardized format - so we can apply one AggregationEngine to all of them regardless of which bureau the data came from
* The output is row-level normalized data (still one row per tradeline) with standardized column names and all the filter flags the AggregationEngine needs to compute its aggregations

# We can see the `transform` method checks if the output features already exist in the data - if they do, it just returns them without running any transformations. This is the pass-through behavior that lets the same pipeline work with both processed data (development) and preprocessed data (production).

In [ ]:
import inspect

print(inspect.getsource(node_dict['trade FE'].transformer.transformers['InputNormalizer'].transform))

In [ ]:
preprocessed.head(5)

In [ ]:
asset_trade_normalizer.keys()

## `info.key = 'ZEST_KEY'` tells the normalizer which column to use as the join key across all tables
## `info.reference_date = 'DATE_OF_REQUEST'` is the anchor date that all time-based calculations are relative to - for example, `months_since_openDate` is the number of months between when the account was opened and when the bureau data was pulled

In [ ]:
asset_trade_normalizer['info']

In [ ]:
type(asset_trade_normalizer['mapping'])

## `mapping` is a list of converter operations that each take a single raw Equifax column and convert it to a standardized name and type - `NumericConverterV2` for numbers (`BALANCE` -> `balance_amt`), `DateConverterV2` for dates (`DATE_OPENED` -> `openDate`), and `StringConverterV2` for strings (`ACCOUNT_TYPE` -> `accountType`). Some also apply value mappings (like `TERMS_FREQUENCY` -> `termFreqMult` where `'M'` becomes `1`, `'W'` becomes `4.33`, etc.)
## Columns marked `drop_from_normalized: True` are intermediate helpers that get used in preprocessing but won't appear in the final normalized output - for example, `PAYMENT_HISTORY_1_24` is needed to compute delinquency patterns but gets dropped after

## high_credit_amt: The highest reported balance for that tradeline
* Installment accounts: if you haven't missed any payments, this should be the balance you started with. If you miss payments, this can accrue on the high credit
* Revolving accounts: the highest balance you had at that point, even if you pay off immediately it would be the most expensive purchase
* See [App Review Guide](https://zestfinance.atlassian.net/wiki/spaces/DS/pages/1710784539/App+Review+Guide) for more details
* We apply a NumericConverter from `HIGH_CREDIT`

## balance_amt: The amount of debt you have right now
* We apply NumericConverter from `BALANCE`, filling NA with 0

## credit_limit: The maximum amount you can borrow
* We first apply a NumericConverter from `CREDIT_LIMIT`, then later in preprocessing we set it to NA for non-revolving, non-open, and non-charge card accounts
* Only makes sense in the context of revolving accounts

## pastDueAmt: The dollar amount on the tradeline that is past due at the report date (may include fees and interest)
* The amount of money you should have paid by now but have not yet
* Revolving: this can be the minimum payments you have missed so far
* Installment: generally equals the sum of the missed payments
* We create from `PAST_DUE_AMOUNT`, filling NA with 0

## scheduled_payment_amount: Contractual amount due for next payment
* Installment: the fixed payment
* Revolving: the minimum payment amount
* We create with NumericConverter from `SCHEDULED_PAYMENT_AMOUNT`
* We later multiply by `termFreqMult` to get `monthly_payment_amt`

## termDur: How long the payment lasts
* The amount of time to repay the loan (page 271 in Equifax docs)
* We create from `TERMS_DURATION`, we do not fill NA with 0 here

## termFreqStr: String version of how often you have to pay

## termFreqMult: How often you have to pay, as a monthly multiplier
* We create from `TERMS_FREQUENCY` using a NumericConverter with a mapping that converts frequency codes into how many months there are per payment period:
```
"M": 1, "B": 2.17, "W": 4.33, "E": 2, "L": 0.5, "Q": 0.33, "S": 0.17, "T": 0.25, "Y": 0.08, "D": 1, "P": 1
```
* Missing values default to 1 (monthly). See page 155 in the Equifax document

## date_of_request: The application date
* Used as the reference date for all time-based calculations. For national data, each person gets a random application date between 0 and 3 months after the latest reported date for their tradelines

## openDate: Date that tradeline was opened
* Created with DateConverter from `DATE_OPENED`

## closedDate: Date that the tradeline was closed
* Equifax: "contains the date the account was closed. It will not be populated when Date Major Delinquency 1st Reported is present"
* Created from `CLOSED_DATE`

## majordqDate: Date of first major delinquency
* Populated if current rate/status is 6 (collection), 7 (chapter 13 bankruptcy), 8 (repossession), 9 (charge off), M, or Z (foreclosure)
* Created from `DMD_REPORTED`

## rptDate: Date the tradeline was last updated

## lstPmtDate: Date the user made the last payment
* Created with DateConverter from `LAST_PAYMENT_DATE`

## accountType: Type of loan
* Created from `ACCOUNT_TYPE` using StringConverter
* See page 150 in the Equifax document for all codes (e.g., 18 = credit card)
* "Contains a code that describes the kind of loan (auto, home improvement, credit card, etc)"

## portfolioType: The type of loan (more general)
* Created from `PORTFOLIO_TYPE` using StringConverter, mapping `O` (open-ended) to `R` (revolving). Open is typically a charge card where you have full payment every cycle

## PORTFOLIO_TYPE: The type of loan (with open kept separate)
* Same column without the O -> R mapping, kept as an intermediate so filters like `is_revolving` and `is_open_ended` can distinguish between them

## ecoa: Relationship of the person to the tradeline
* Created from `ECOA_DESIGNATOR` using StringConverter
* In the preprocessing step, we drop rows where ecoa is `A` (authorized user), `T` (terminated), or `X` (deceased) since these people are not financially responsible for the tradeline

### Account Designator Codes (ECOA)

| CODE | DESCRIPTION |
| :---: | :--- |
| A | Authorized User - another individual has contractual responsibility |
| B | On behalf of another person - the subject has financial responsibility but the account is used exclusively by another person |
| C | Co-maker - the subject has co-signed for a loan and will be responsible if the borrower defaults |
| I | Individual Account - the subject is primarily responsible for payment |
| J | Joint Account - the subject and another person are jointly responsible for payment |
| M | Maker - the subject is responsible for payment, but a co-maker will be responsible if maker defaults |
| S | Shared, but otherwise undesignated - credit grantor knows the subject and at least one other person share the account but not enough info to designate as J or A |
| T | Terminated - the subject's relationship to this account has ended |
| U | Undesignated |
| X | Deceased (not returned on trade lines) |

## NARRATIVE_CODE_1 and NARRATIVE_CODE_2
* Both created using StringConverter
* These are comment codes that indicate certain conditions about the tradeline (e.g., `BZ` = account paid for less than full balance, `CB` = charged off, `IL` = chapter 7 bankruptcy, `DO` = chapter 13 bankruptcy)
* Both codes can carry the same types of values - a tradeline can have up to two narrative codes at once
* These are used heavily in the preprocessing filters (e.g., `is_bnkr`, `is_closed`, `is_chargeoff`, `is_negative`) to identify account statuses that the `RATE_STATUS_CODE` alone doesn't capture

## dqDate: Date of the highest delinquency that occurred outside the payment history window
* Created from `PREVIOUS_HIGH_DATE_1` using DateConverter
* The payment history covers the most recent 48 months. `dqDate` captures any delinquency that happened before that window
* Used in the `is_never_dq` filter - if `dqDate` is NA and there are no delinquencies in the 48-month payment history, the tradeline has never been delinquent

## RATE_STATUS_CODE: The current rating on the account
* Rate codes are numeric, status codes are letters
* Created using StringConverter, used as an intermediate (dropped from normalized output)

### Rate Codes

| CODE | DESCRIPTION |
| :---: | :--- |
| 0 | Too new to rate; approved but not used |
| 1 | Pays account as agreed |
| 2 | Not more than two payments past due (DQ30) |
| 3 | Not more than three payments past due (DQ60) |
| 4 | Not more than four payments past due (DQ90) |
| 5 | At least 120 days or more than four payments past due (DQ120) |
| 6 | Collection account |
| 7 | Included in Chapter 13 |
| 8 | Repossession |
| 9 | Charge-off |

### Status Codes

| CODE | DESCRIPTION |
| :---: | :--- |
| A | Account is inactive |
| B | Lost or stolen card |
| C | Contact member for status |
| D | Refinanced or renewed |
| E | Consumer deceased |
| G | Foreclosure process started |
| M | Included in Chapter 13 |
| S | Dispute - resolution pending |
| Z | Included in Bankruptcy |
| # | In bankruptcy of another person (retired 2009) |

## Payment_History_1_24, Payment_History_25_36, Payment_History_37_48
* The payment history strings covering the most recent 48 months. Leftmost character is most recent month
* Created using StringConverter, used as intermediates (dropped from normalized output)
* The `PaymentPatternsAggregatorV2` preprocessor combines all three strings, trims to 48 months, and extracts trended features like `number_DQ30+_24_months`, `number_CO_12_months`, and `months_since_DQ60+`

## ACTIVITY_DESIGNATOR: The final state of the account
* Created using StringConverter, used as an intermediate (dropped from normalized output)

| Code | Description |
| :---: | :--- |
| B | Paid and Closed |
| C | Closed |
| D | Transfer/Sold/Paid |
| L | Lost/Stolen |
| P | Paid |
| R | Refinanced |
| T | Transfer/Sold |

* Used in the `is_closed` and `is_transfer` filters

## chargeoff_amt: The amount the creditor wrote off when the account was charged off
* Created from `ORIGINAL_CHARGE_OFF_AMOUNT` using NumericConverter
* In preprocessing, if `high_credit_amt` is NA we replace it with `chargeoff_amt` via the `CoalesceV2` step - our best guess for the original loan amount when the high credit is missing

## CUSTOMER_NUMBER: The industry code / kind of business
* Created using StringConverter, used as an intermediate (dropped from normalized output)
* Used to create the `is_member` filter where `CUSTOMER_NUMBER == 'FC'` identifies credit union tradelines

## constant_placeholder: A constant column of 1s for every row
* Created from `ZEST_KEY` using StringConverter with `replace_all_with_constant: '1'`
* Used in the AggregationEngine for count aggregations so that null values in other columns don't affect the count



## We can see that our InputNormalizer has the mapper transformer, preprocess transformer, and the post process transformer

In [ ]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

## We can see our Mapper is a [MapperV2](https://github.com/Katlean/model-engine/blob/master/model_engine/feature_engine_V2/listed_objects_engines.py#L135) object which takes in the mapping asset as an input

It is called [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/feature_engine_V2/feature_engine.py#L186) in InputNormalizer
```python
mapper = MapperV2(api=self.asset['mapping'])
```

* `MapperV2` takes the mapping list from the asset and converts each dict into an actual transformer object (e.g., `NumericConverterV2`, `DateConverterV2`, `StringConverterV2`) stored in an `OrderedDict` - the order matters because each transformer's output is fed as input to the next
* When `transform(data)` is called, it loops through each transformer in order and applies it to the data, progressively renaming and converting raw bureau columns into their canonical forms

In [ ]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers']

In [ ]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

## These individual transformers (NumericConverterV2, DateConverterV2, FilterV2, etc.) live in the [feature-engine-parts](https://github.com/Katlean/feature-engine-parts/tree/master/feature_engine_parts/fe_parts_V2) codebase

* If the asset is the blueprint and the InputNormalizer is the foreman, then `feature-engine-parts` are the actual tools and workers - each converter, filter, and aggregator is a lightweight transformer class that does one specific job. As [Jackie put it](https://drive.google.com/file/d/1cTEUrDFCauPTh2Fhs5hYMoiFQ1IKl9Me/view): "feature-engine-parts are all the smaller bricks, model-engine is the design that tells you how to use those bricks to build a house"


In [ ]:
mapper_dict = {}
for mapper in node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers'].stack:
    name = mapper.transformer_name
    print(f'mapper: {mapper} has transformer: {mapper.transformer_name}')
    print(f'raw_feature input is {mapper.raw_feature}')
    print(f'output is {mapper.new_feature}')
    mapper_dict[name] = mapper


In [ ]:
mapper_dict['PAYMENT_HISTORY_1_24'].__dict__

In [ ]:
mapped_df = node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers'].transform(preprocessed)

In [ ]:
mapped_df_inq = node_dict['inq FE'].transformer.transformers['InputNormalizer'].transformers['Mappers'].transform(preprocessed_inq)

### What this is showing

We are checking whether inquiry records fall into the expected **gap window** between the applicant's bureau archive date and their application date.

#### The two checks
- `is_before_inq = inquiry_date < appDate`  
  Counts inquiries that happened **before the application date**.
- `inquiry_date_after_archive = inquiry_date >= ARCHIVE_DATE`  
  Counts inquiries that happened **on or after the bureau archive date**.

#### Why this matters
If an inquiry satisfies **both** conditions, then it falls in the missing window:

`ARCHIVE_DATE <= inquiry_date < appDate`

That is the period after the quarterly bureau archive was pulled, but before the applicant actually applied. These are the inquiries that would be missing from the base archive and need to be added from the next quarter pull.

#### Why some inquiries are after `ARCHIVE_DATE`
This is expected. The archive is only a quarterly snapshot, and the applicant may apply later. Some inquiries can happen in between those two dates.

So an inquiry being **after `ARCHIVE_DATE`** is not leakage by itself. It is still valid as long as it is also **before `appDate`**.

The real leakage risk would be inquiries with:

`inquiry_date >= appDate`

#### What the counts mean
The output:

```python
(7524, 351)

In [ ]:
archive_counts = (
    mapped_df.groupby("ZEST_KEY")["ARCHIVE_DATE"]
    .nunique()
)
max_archive_count = archive_counts.max()

print("Max distinct ARCHIVE_DATE per ZEST_KEY:", max_archive_count)

In [ ]:
archive_counts_inq = (
    mapped_df_inq.groupby("ZEST_KEY")["ARCHIVE_DATE"]
    .nunique()
)
max_archive_count_inq = archive_counts_inq.max()

print("Max distinct ARCHIVE_DATE per ZEST_KEY for INQ:", max_archive_count_inq)

In [ ]:
is_before_inq = mapped_df_inq['inquiry_date'] < mapped_df_inq['appDate']

In [ ]:
is_before_inq.sum()

In [ ]:
inquiry_date_after_archive = mapped_df_inq['inquiry_date'] >= mapped_df_inq['ARCHIVE_DATE']

# We don't have this for trade!

In [ ]:
( mapped_df['rptDate'] >= mapped_df['ARCHIVE_DATE']).sum()

# We can see the T6 here: 

In [ ]:
is_before_inq.sum(), inquiry_date_after_archive.sum()

In [ ]:
mapped_df

In [ ]:
mapped_df[['rptDate', 'appDate']].dtypes

In [ ]:
is_before = mapped_df['rptDate'] < mapped_df['appDate']

In [ ]:
is_before.sum()

In [ ]:


is_before = mapped_df['rptDate'] < mapped_df['appDate']

# Check if this condition is True for ALL rows in the dataframe
all_quarters_before = is_before.all()

print(f"Does the quarter always end before the appDate? {all_quarters_before}")


if not all_quarters_before:
    print("\nHere are the rows where the quarter does NOT occur before the appDate:")
    violations = all_preprocessed[~is_before]
    display(violations[['quarter', 'appDate']])

# Preprocessing

In [ ]:
asset_trade_normalizer['preprocess']

In [ ]:
preprocess_dict = {}
for preprocessor in node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Preprocessors'].stack:
    name = preprocessor.transformer_name
    print(f'preprocessor: {preprocessor} ')
    print(f'raw_feature input is {preprocessor.input_dtypes}')
    try:
        print(f'output is {preprocessor.new_feature}')
    except:
        print(f'output is {preprocessor.output_dtypes}')
    preprocess_dict[name] = preprocessor


In [ ]:
preprocess_dict.keys()

# Preprocessing

After the Mapper converts raw columns to canonical names, the Preprocessor creates derived features and boolean filter flags.



## Date Differences

From the [preprocess asset](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/equifax/cms_6/fe2/trade.json#L261-L285), the first three entries are `DateDiffV2` operations:
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'rptDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_rptDate'}}
```
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'openDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_openDate'}}
```
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'lstPmtDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_lstPmtDate'}}
```
##  Each computes the number of months between `date_of_request` (the reference date from `info`) and the respective date. These tell us how recently the tradeline was reported, how old the account is, and when the last payment was made.


## Payment History Features ([PaymentPatternsAggregatorV2](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py))

This is the most complex preprocessor. The asset entry looks like:
```python
{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    ...
    'DQ30': ['2'], 'DQ60': ['3'], 'DQ90': ['4']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+', 'DQ60+', 'DQ90+', 'DQ120+', 'CO', 'DQ30', 'DQ60', 'DQ90']}}}
```

We can see the transform [here](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L214-L230)

```python
 def transform(self, data):
        data = data.copy(deep=True)
        input_columns = list(data.columns)
        data[self.zest_ppt_name] = self._construct_payment_pattern_cols(data)
        data[self.ppt_len_name] = self._get_effective_month_range(
            data[self.zest_ppt_name], data[self.zest_ppt_name].str.len(), exclude_trailing=["*"]
        )
        data = self._construct_trended_features(data, data[self.zest_ppt_name])
        data = self._construct_months_since_features(data)
        output_columns = input_columns + self._new_features
        if self.return_zest_ppt:
            output_columns.append(self.zest_ppt_name)
        if self.return_ppt_len:
            output_columns.append(self.ppt_len_name)
        data = data[output_columns]
        super().enforce_dtypes(data)
        return data
```

### Step 1: Construct the zest_payment_pattern ([`_construct_payment_pattern_cols`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L152))
```python
def _construct_payment_pattern_cols(self, data):
    new_ppt = self._combine_payment_patterns(data)
    new_ppt = self._remove_placeholder(new_ppt)
    new_ppt = self._trim(new_ppt)
    new_ppt = self._add_fillers(data, new_ppt)
    return new_ppt

**1a) Combine** all pattern strings left to right ([`_combine_payment_patterns`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L106)):
```python
def _combine_payment_patterns(self, data):
        """
        Combines different payment patterns to a single payment pattern
        """
        new_ppt = None
        first_ppt_name = self.patterns[0]
        new_ppt = data[first_ppt_name].copy(deep=True)
        for ppt in self.patterns[1:]:
            ppt_data_ = data[ppt].copy(deep=True)
            ppt_data_.loc[ppt_data_.isna()] = ""
            new_ppt += ppt_data_
        return new_ppt

In [ ]:
asset_trade_normalizer['preprocess'][3]

## 1a) Append patterns left to right so the most recent is on the left
* `RATE_STATUS_CODE` (current month) + `PAYMENT_HISTORY_1_24` (months 1-24) + `PAYMENT_HISTORY_25_36` + `PAYMENT_HISTORY_37_48` - the combined string reads newest to oldest left to right

In [ ]:
data_for_payment_columns = mapped_df.copy()
patterns = asset_trade_normalizer['preprocess'][3]['params']['payment_patterns']['patterns']

In [53]:
## running_combine_payment patterns

new_ppt = None
first_ppt_name = patterns[0]

new_ppt = data_for_payment_columns[first_ppt_name].copy(deep=True)
print(f"first pattern: {first_ppt_name} gives us new_ppt: {new_ppt.head(3).values}")
for ppt in patterns[1:]:

    ppt_data_ = data_for_payment_columns[ppt].copy(deep=True)
    print(f'for {ppt} we have ppt_data_: {ppt_data_.head(3).values}')
    ## make na ones blank
    ppt_data_.loc[ppt_data_.isna()] = ""
    ### append to the right
    new_ppt += ppt_data_
    print(f'After adding {ppt} we have new_ppt: {new_ppt.head(3).values}')



first pattern: RATE_STATUS_CODE gives us new_ppt: ['1' '1' '6']
for PAYMENT_HISTORY_1_24 we have ppt_data_: ['111111111111/111111111111' 'EEEEEEEEE111/111111111111'
 '************/************']
After adding PAYMENT_HISTORY_1_24 we have new_ppt: ['1111111111111/111111111111' '1EEEEEEEEE111/111111111111'
 '6************/************']
for PAYMENT_HISTORY_25_36 we have ppt_data_: ['/1EEEEEEEEE*1' '/1111111111*1' '/************']
After adding PAYMENT_HISTORY_25_36 we have new_ppt: ['1111111111111/111111111111/1EEEEEEEEE*1'
 '1EEEEEEEEE111/111111111111/1111111111*1'
 '6************/************/************']
for PAYMENT_HISTORY_37_48 we have ppt_data_: ['/111EEEEEEEE1' '/111111111111' '/************']
After adding PAYMENT_HISTORY_37_48 we have new_ppt: ['1111111111111/111111111111/1EEEEEEEEE*1/111EEEEEEEE1'
 '1EEEEEEEEE111/111111111111/1111111111*1/111111111111'
 '6************/************/************/************']


In [54]:
data_for_payment_columns[data_for_payment_columns[first_ppt_name]=="*"].index

Index([348, 1175, 1901, 2880, 2881], dtype='int64')

In [55]:
new_ppt[348]

'*************/************/************/************'

This concatenates `RATE_STATUS_CODE` + `PAYMENT_HISTORY_1_24` + `PAYMENT_HISTORY_25_36` + `PAYMENT_HISTORY_37_48` into one string. The `RATE_STATUS_CODE` goes first (leftmost = most recent), then months 1-24, then 25-36, then 37-48.

**1b) Remove placeholders** ([`_remove_placeholder`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L134-L143)): Equifax uses `/` every 12 months as a divider:
```python
def _remove_placeholder(self, new_ppt):
        """
        Removes placeholders if they exist

        i.e. in Equifax '/' is a placeholder for every 12 months of history, it is removed for accuracy
        """
        if self.placeholder:
            new_ppt = new_ppt.str.replace(self.placeholder, "")
        return new_ppt

```

In [56]:
placeholder = asset_trade_normalizer['preprocess'][3]['params']['payment_patterns']['placeholder']
print(f'placeholder: {placeholder}')

placeholder: /


In [57]:
new_ppt.head(3)

0    1111111111111/111111111111/1EEEEEEEEE*1/111EEE...
1    1EEEEEEEEE111/111111111111/1111111111*1/111111...
2    6************/************/************/******...
Name: RATE_STATUS_CODE, dtype: object

In [58]:
new_ppt = new_ppt.str.replace(placeholder, "")

In [59]:
new_ppt.head(3)

0    11111111111111111111111111EEEEEEEEE*1111EEEEEEEE1
1    1EEEEEEEEE1111111111111111111111111*1111111111111
2    6************************************************
Name: RATE_STATUS_CODE, dtype: object

## We then trim the data so we only keep 2 years of data: 48 characters




**1c) Trim** to 48 characters so it matches production length ([`_trim`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L144-L151)):
```python
def _trim(self, new_ppt):
    if self.trim:
        new_ppt = new_ppt.str[:self.trim]
    return new_ppt
```

In [60]:
trim = asset_trade_normalizer['preprocess'][3]['params']['payment_patterns']['trim']

In [61]:
new_ppt.head(3)

0    11111111111111111111111111EEEEEEEEE*1111EEEEEEEE1
1    1EEEEEEEEE1111111111111111111111111*1111111111111
2    6************************************************
Name: RATE_STATUS_CODE, dtype: object

In [62]:
new_ppt = new_ppt.str[:trim]

In [63]:
new_ppt.head(3)

0    11111111111111111111111111EEEEEEEEE*1111EEEEEEEE
1    1EEEEEEEEE1111111111111111111111111*111111111111
2    6***********************************************
Name: RATE_STATUS_CODE, dtype: object


**1d) Add fillers** ([`_add_fillers`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L119)) - prepend `#` characters to the left to account for the gap between the report date and the application date:
```python
   def _add_fillers(self, data, new_ppt):
        """
        Adds a filler to represent the gap between the most recent report date
        and the current payment pattern. The filler is a '#'
        This is enables quick calculation of trended features
        """
        if self.report_date:
            months_since_ppt = (
                np.ceil(data[self.date_delta_col]).fillna(0).astype(int)
            )  # if missing, assume updated today
            z = pd.Series(["#"] * len(months_since_ppt), index=data.index, dtype="str")
            filler = pd.Series(z).str.repeat(repeats=months_since_ppt.astype(int)).astype("str")
            new_ppt = filler.str.cat(new_ppt, join="left")  # append the filler
        return new_ppt
```

Note that `self.date_delta_col` is `months_since_rptDate` (set from `report_date: 'rptDate'` in the asset). So if the tradeline was reported 2 months before the application date, we prepend `##` to the front. This way position 0 always represents the application month.

* The bureau data is a quarterly snapshot. `rptDate` is when the tradeline was last reported, which could be 1-3 months before the application date
* Without fillers, position 0 would be "month the tradeline was reported." We want position 0 to be "month of application" so all time windows are relative to when the person applied
* We prepend `#` characters equal to `months_since_rptDate` to shift everything over. If the app was in March and the tradeline was reported in January, we add `##` so position 0 = March, position 2 = January, etc.
* The `#` characters don't count as valid history - `_get_effective_month_range` subtracts them from the denominator so they don't affect percent calculations

## Note for this, we need to create months_since_rptDate : We can confirm because it is in the input_dtypes of the object preprocess_dict['PaymentPatternsAggregatorV2'].input_dtypes

In [64]:
data_for_payment_columns = preprocess_dict['months_since_rptDate'].transform(data_for_payment_columns)

In [65]:
preprocess_dict['PaymentPatternsAggregatorV2'].input_dtypes

{'RATE_STATUS_CODE': 'string',
 'PAYMENT_HISTORY_1_24': 'string',
 'PAYMENT_HISTORY_25_36': 'string',
 'PAYMENT_HISTORY_37_48': 'string',
 'months_since_rptDate': 'numeric'}

In [66]:
preprocess_dict['months_since_rptDate'].__dict__

{'_init_args': (),
 '_init_kwargs': {'feature': 'rptDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_rptDate'},
 'transformer_name': 'months_since_rptDate',
 '_other_params': {},
 'input_dtypes': {'rptDate': 'date', 'date_of_request': 'date'},
 'output_dtypes': {'months_since_rptDate': 'numeric'},
 'feature': 'rptDate',
 'reference_feature': 'date_of_request',
 'new_feature': 'months_since_rptDate',
 'units': 'M'}

In [67]:
data_for_payment_columns[['months_since_rptDate', 'rptDate', 'date_of_request']].head(10)

,months_since_rptDate,rptDate,date_of_request
0,0.262839,2023-03-20,2023-03-28
1,0.262839,2023-03-20,2023-03-28
2,0.821372,2023-03-03,2023-03-28
3,0.919937,2023-02-28,2023-03-28
4,0.821372,2023-03-03,2023-03-28
5,1.018501,2023-02-25,2023-03-28
6,71.097969,2017-04-24,2023-03-28
7,0.919937,2023-02-28,2023-03-28
8,0.197129,2023-03-22,2023-03-28
9,0.591388,2023-03-10,2023-03-28


In [68]:
data_for_payment_columns[['months_since_rptDate', 'rptDate', 'date_of_request']].dtypes

months_since_rptDate           float64
rptDate                 datetime64[ns]
date_of_request         datetime64[ns]
dtype: object

In [69]:
date_delta_col = 'months_since_rptDate'

## We can see for this one below if we didn't add the filler we would think all of their payment patterns were recent

In [70]:
data_for_payment_columns[['months_since_rptDate', 'rptDate', 'date_of_request']].iloc[6,:]

months_since_rptDate              71.097969
rptDate                 2017-04-24 00:00:00
date_of_request         2023-03-28 00:00:00
Name: 6, dtype: object

In [71]:
new_ppt[6]

'9**555432***************************************'

## We use `np.ceil` to always round up - if 1.5 months since report date, we add 2 `#` fillers rather than 1, because we don't have reliable data for that partial month so we skip it rather than counting it as valid payment history

In [72]:
months_since_ppt = np.ceil(data_for_payment_columns[date_delta_col]).fillna(0).astype(int)

In [73]:
months_since_ppt

0         1
1         1
2         1
3         1
4         1
       ... 
9718     79
9719     81
9720     95
9721    104
9722    110
Name: months_since_rptDate, Length: 9723, dtype: int64

# We then create a serires of # to append to the left of our payment pattern to account for the time between the report date of the tradeline and the person's application date

In [74]:
# first one just creates a seiries with as many rows as we have 
z = pd.Series(["#"] * len(months_since_ppt), index=data_for_payment_columns.index, dtype="str")

## this one fills out
filler = pd.Series(z).str.repeat(repeats=months_since_ppt.astype(int)).astype("str")

In [75]:
z

0       #
1       #
2       #
3       #
4       #
       ..
9718    #
9719    #
9720    #
9721    #
9722    #
Length: 9723, dtype: object

In [76]:
filler

0                                                       #
1                                                       #
2                                                       #
3                                                       #
4                                                       #
                              ...                        
9718    ##############################################...
9719    ##############################################...
9720    ##############################################...
9721    ##############################################...
9722    ##############################################...
Length: 9723, dtype: object

In [77]:
filler[6] 

'########################################################################'

In [78]:
new_ppt.head(7)

0    11111111111111111111111111EEEEEEEEE*1111EEEEEEEE
1    1EEEEEEEEE1111111111111111111111111*111111111111
2    6***********************************************
3    1111111111111111111111111***********************
4    211*********************************************
5    6***********************************************
6    9**555432***************************************
Name: RATE_STATUS_CODE, dtype: object

In [79]:
new_ppt = filler.str.cat(new_ppt, join="left")  # append the filler

## We can now see that one with 72 months in between has tons of #

In [80]:
new_ppt.head(7)

0    #11111111111111111111111111EEEEEEEEE*1111EEEEEEEE
1    #1EEEEEEEEE1111111111111111111111111*111111111111
2    #6***********************************************
3    #1111111111111111111111111***********************
4    #211*********************************************
5    ##6*******************************************...
6    ##############################################...
dtype: object

## We then compute the effective month range of history that is the number of months in that window where they actually had history (excluding the filler we created)


### Step 2: Compute payment_history_length ([`_get_effective_month_range`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L166))

Not all characters are valid history. `#` means "before reporting started" and `*` means "break in coverage." Called in `transform`:
```python
data[self.ppt_len_name] = self._get_effective_month_range(
    data[self.zest_ppt_name], 
    data[self.zest_ppt_name].str.len(), 
    exclude_trailing=['*'])
```
```python
def _get_effective_month_range(self, trimmed, month_range, exclude_trailing=[]):
    z_count = trimmed.str.count('#')
    if len(exclude_trailing) > 0:
        z_count = z_count + trimmed.apply(
            lambda x: self._count_trailing_matches(str(x), exclude_trailing))
    effective_month_range = month_range - z_count
    return effective_month_range
```

In [81]:
trimmed = new_ppt

# Note, trailing `*` characters at the end of the payment string mean the person's credit history doesn't go back that far. Those months never existed for them, so we subtract them from the denominator. However, a `*` in the middle of the string means there was a gap in reporting but time still passed. We keep those in the denominator because the month still elapsed even if we don't have the status for it.

## We want the denominator of the variables we create to reflect months that actually passed since the person opened this tradeline. Trailing `*`s on the right mean the tradeline didn't exist yet back then so we don't count them. `#`s on the left mean it hasn't been updated since the report date so we don't count those either. `*`s in the middle mean we don't have data for that month but the tradeline still existed so we still count those as months elapsed.

In [82]:
exclude_trailing = ['*']

```python
def _count_trailing_matches(self, input_str, char_list):
        count = 0
        for char in reversed(input_str.strip()):
            if char in char_list:
                count += 1
            else:
                break
        return count
```

In [83]:
def _count_trailing_matches(input_str, char_list):
        count = 0
        for char in reversed(input_str.strip()):
            if char in char_list:
                count += 1
            else:
                break
        return count

In [84]:
for i, val in enumerate(trimmed.head(10)):
    trailing_count = 0
    for char in reversed(str(val).strip()):
        if char == '*':
            trailing_count += 1
        else:
            break
    print(f'Row {i}: trailing *s = {trailing_count}, pattern = ...{str(val)[-20:]}')


Row 0: trailing *s = 0, pattern = ...EEEEEEE*1111EEEEEEEE
Row 1: trailing *s = 0, pattern = ...1111111*111111111111
Row 2: trailing *s = 47, pattern = ...********************
Row 3: trailing *s = 23, pattern = ...********************
Row 4: trailing *s = 45, pattern = ...********************
Row 5: trailing *s = 47, pattern = ...********************
Row 6: trailing *s = 39, pattern = ...********************
Row 7: trailing *s = 47, pattern = ...********************
Row 8: trailing *s = 0, pattern = ...11111111111111111111
Row 9: trailing *s = 0, pattern = ...11111111111111111111


In [85]:
z_count = trimmed.str.count('#')
trimmed_count = trimmed.apply(lambda x: _count_trailing_matches(str(x), exclude_trailing))

In [86]:
z_count

0         1
1         1
2         1
3         1
4         1
       ... 
9718     79
9719     81
9720     95
9721    104
9722    110
Length: 9723, dtype: int64

In [87]:
trimmed_count

0        0
1        0
2       47
3       23
4       45
        ..
9718    47
9719    47
9720    47
9721    47
9722    47
Length: 9723, dtype: int64

In [88]:
new_ppt

0       #11111111111111111111111111EEEEEEEEE*1111EEEEEEEE
1       #1EEEEEEEEE1111111111111111111111111*111111111111
2       #6***********************************************
3       #1111111111111111111111111***********************
4       #211*********************************************
                              ...                        
9718    ##############################################...
9719    ##############################################...
9720    ##############################################...
9721    ##############################################...
9722    ##############################################...
Length: 9723, dtype: object

In [89]:
z_count = z_count + trimmed_count

# We set the total month range as the length of the payment pattern string and the effective month history is that minus the z_count: all the months that aren't real tradeline history. We exclude trailing `*`s (tradeline hadn't been opened yet) and `#`s (months between the tradeline's last update and the application date). We keep middle `*`s because even though no data was reported that month, the tradeline was still open so time still elapsed.

In [90]:
month_range = new_ppt.str.len()

In [91]:
effective_month_range = month_range - z_count

In [92]:
effective_month_range[348]

0


So: `payment_history_length = total_string_length - #_count - trailing_*_count`

### Step 3: Construct trended features [`_construct_trended_features`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L183)

For each time window `[3, 6, 12, 24, 48]` months, we trim the string to that window and count delinquency occurrences:
```python
   def _construct_trended_features(self, data, new_ppt):
        for month_range in self.month_ranges:
            trimmed = new_ppt.str[:month_range]
            effective_month_count = self._get_effective_month_range(trimmed, month_range)
            for rate, values in self._rate.items():
                count = self._get_count(trimmed, values)
                data[f"number_{rate}_{month_range}_months{self.name}"] = count.astype(self.PANDAS_DTYPES["numeric"])
                data[f"percent_{rate}_{month_range}_months{self.name}"] = (count / effective_month_count).astype(
                    self.PANDAS_DTYPES["numeric"]
                )
        return data
```
Where [`_get_count`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L159-L164) counts how many characters in the trimmed string match any of the rate's values:
```python
 def _get_count(self, trimmed, values):
        if len(values) == 1:
            return trimmed.str.count(values[0])
        else:
            pattern = "|".join(values) ## use regex to find the pattern
            return trimmed.str.count(pattern)

```
For example with `DQ30+: ['2','3','4','5','6','7','8','9','G','K','L','Z']` and `month_range=12`, it trims the string to the first 12 characters, counts how many match any of those values, and produces `number_DQ30+_12_months` and `percent_DQ30+_12_months` (count divided by effective months in that window).

In [93]:
 def _get_count(trimmed, values):
    if len(values) == 1:
        return trimmed.str.count(values[0])
    else:
        pattern = "|".join(values)
        return trimmed.str.count(pattern)


In [94]:
asset_trade_normalizer['preprocess'][3]['params']['payment_patterns']['rate']



{'paid_as_agreed': ['0', '1'],
 'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ30': ['2'],
 'DQ60': ['3'],
 'DQ90': ['4'],
 'DQ120': ['5']}

In [95]:
asset_trade_normalizer['preprocess'][3]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ30': ['2'],
    'DQ60': ['3'],
    'DQ90': ['4'],
    'DQ120': ['5']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}}}

In [96]:
rate = asset_trade_normalizer['preprocess'][3]['params']['payment_patterns']['rate']

In [97]:
type(rate)

dict

In [98]:
type(asset_trade_normalizer['preprocess'][3])

dict

In [99]:
rate.items()

dict_items([('paid_as_agreed', ['0', '1']), ('DQ30+', ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z']), ('DQ60+', ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z']), ('DQ90+', ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z']), ('DQ120+', ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z']), ('CO', ['6', '8', '9', 'G', 'K', 'L', 'Z']), ('DQ30', ['2']), ('DQ60', ['3']), ('DQ90', ['4']), ('DQ120', ['5'])])

## We can see for the _get_effective_month_range in [_construct_trended_features](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L183), we do not include the trailing in effective month range. This means the effective month range is only excluding the count of #. 

### So if if someones entire string was say ##101******************.

### Then count_paid_as_agreed_6 would first look at ##101* and see it would be 3/4 months where they paid as agreed. 



``python
def _construct_trended_features(self, data, new_ppt):
        for month_range in self.month_ranges:
            trimmed = new_ppt.str[:month_range]
            effective_month_count = self._get_effective_month_range(trimmed, month_range)
            for rate, values in self._rate.items():
                count = self._get_count(trimmed, values)
                data[f"number_{rate}_{month_range}_months{self.name}"] = count.astype(self.PANDAS_DTYPES["numeric"])
                data[f"percent_{rate}_{month_range}_months{self.name}"] = (count / effective_month_count).astype(
                    self.PANDAS_DTYPES["numeric"]
                )
        return data
```

In [100]:
month_ranges = preprocess_dict['PaymentPatternsAggregatorV2'].month_ranges
for month_range in month_ranges:
    ## we subset to only the most recent month range
    if month_range == 6:
        should_print = True
    else:
        should_print = False
    trimmed = new_ppt.str[:month_range]
    if should_print:
        print(f'trimmed.head(5): {trimmed.head(3).values}, for month_range: {month_range}')

    ## we then get the effective_month count for that range
    z_count = trimmed.str.count('#')
    if should_print:
        print(f'z_count for month_range {month_range} is {z_count.head(3).values}')
    ## We don't use trailing for this
    #trimmed_count = trimmed.apply(lambda x: _count_trailing_matches(str(x), exclude_trailing))
    #if should_print:
        #print(f'trimmed_count for month_range {month_range} is {trimmed_count.head(3).values}')
    effective_month_count = trimmed.str.len() - (z_count)
    if should_print:
        print(f'effective_month_count for month_range {month_range} is {effective_month_count.head(3).values}')
    for this_rate, values in rate.items():
        if this_rate == 'paid_as_agreed':
            should_print_rate = True
        else:
            should_print_rate = False
        if should_print_rate and should_print:
            print(f'rate: {this_rate}, values: {values}')
        count =_get_count(trimmed, values)
        if should_print_rate and should_print:
            print(f'count for rate {this_rate} and  {count[0:3].values}')
        ## no name
        perc = count/effective_month_count
        data_for_payment_columns[f"number_{this_rate}_{month_range}_months"]  = count
        data_for_payment_columns[f"percent_{this_rate}_{month_range}_months"]  = perc
        if should_print_rate and should_print:
            print(f'percent for rate {this_rate} and  {perc[0:3].values}')
    

trimmed.head(5): ['#11111' '#1EEEE' '#6****'], for month_range: 6
z_count for month_range 6 is [1 1 1]
effective_month_count for month_range 6 is [5 5 5]
rate: paid_as_agreed, values: ['0', '1']
count for rate paid_as_agreed and  [5 1 0]
percent for rate paid_as_agreed and  [1.  0.2 0. ]


### Step 4: Construct months_since features ([`_construct_months_since_features`](https://github.com/Katlean/feature-engine-parts/blob/b02065e09502f828b5d364a99f8d457b93bd77d1/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L195))

For each rate level, find the position of the most recent occurrence in the full payment pattern string:
```python
    def _construct_months_since_features(self, data):
        # convert to contiguous NumPy string array (faster than object dtype)
        col = data[self.zest_ppt_name].astype("string").to_numpy(dtype="str")

        for rate, values in self.rate.items():
            # build a (len(values) × len(col)) matrix of character positions
            positions = np.char.find(col[None, :], np.array(values, dtype="str")[:, None]).astype(float)

            # replace “not-found” (-1) with NaN
            positions[positions == -1] = np.nan
            # take the earliest position for each value
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore")
                earliest_pos = np.nanmin(positions, axis=0)

            data[f"months_since_most_recent_{rate}{self.name}"] = earliest_pos

        return data
```
* For each rate (e.g., DQ60+), we search the payment pattern string for every character in that rate's values list. `np.char.find` returns the first position each character appears at, or -1 if not found. We replace -1 with NaN so it doesn't win the min, then take the minimum position across all characters to get the most recent occurrence
* Since the string reads left to right from most recent to oldest, the position index directly gives us the months since. Example: string is `##11132111...`, looking for `3`, found at position 5, so `months_since_most_recent_DQ60+ = 5`

In [101]:
rate

{'paid_as_agreed': ['0', '1'],
 'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
 'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
 'DQ30': ['2'],
 'DQ60': ['3'],
 'DQ90': ['4'],
 'DQ120': ['5']}

In [102]:
## just run for paid as agreed first
import warnings
col = new_ppt.astype("string").to_numpy(dtype="str")
df_dict = {}
for this_rate, values in rate.items():
    # build a (len(values) × len(col)) matrix of character positions
    positions = np.char.find(col[None, :], np.array(values, dtype="str")[:, None]).astype(float)
    positions[positions == -1] = np.nan ## we replace with nana 
            # take the earliest position for each value
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        earliest_pos = np.nanmin(positions, axis=0)
    data_for_payment_columns[f"months_since_most_recent_{this_rate}"] = earliest_pos
    df_dict[this_rate] = {'positions': positions, 'earliest_pos': earliest_pos, 'values': values}

## We build a matrix where each row is one of the characters we're looking for and each column is a person. Each cell holds the first index where that character appeared in that person's payment pattern. We then take the minimum across all rows for each person to find the most recent occurrence of any of those characters - that index is the months since that rate last occurred.

In [103]:
data_for_payment_columns[f"months_since_most_recent_DQ30+"]

0       NaN
1       NaN
2       1.0
3       NaN
4       1.0
       ... 
9718    NaN
9719    NaN
9720    NaN
9721    NaN
9722    NaN
Name: months_since_most_recent_DQ30+, Length: 9723, dtype: float64

# We can see the second person 

In [104]:
df_dict['DQ30+']['values']

['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z']

# We can see there earliest position is 1. It has been 1 month since they had a 6 code: which is inside DQ60+ values

In [105]:
new_ppt[2]

'#6***********************************************'

# The index for the 6 value is 1 the rest are NAN

In [106]:
df_dict['DQ30+']['positions'][:, 2]

array([nan, nan, nan, nan,  1., nan, nan, nan, nan, nan, nan, nan])

In [107]:
df_dict['DQ30+']['positions'].shape

(12, 9723)

In [108]:
data_for_payment_columns[f'months_since_most_recent_DQ30+'].head(5)

0    NaN
1    NaN
2    1.0
3    NaN
4    1.0
Name: months_since_most_recent_DQ30+, dtype: float64

## Note, to run the PaymentPatternsAggregator we need to run rptDate first! See the input_dtypes!

In [109]:
preprocess_dict['PaymentPatternsAggregatorV2'].input_dtypes

{'RATE_STATUS_CODE': 'string',
 'PAYMENT_HISTORY_1_24': 'string',
 'PAYMENT_HISTORY_25_36': 'string',
 'PAYMENT_HISTORY_37_48': 'string',
 'months_since_rptDate': 'numeric'}

In [110]:
preprocess_dict.keys()

dict_keys(['months_since_rptDate', 'months_since_openDate', 'months_since_lstPmtDate', 'PaymentPatternsAggregatorV2', 'high_credit_amt', 'termDur', 'credit_limit', 'credit_limit_2', 'blnc_to_hc', 'hc_to_cl', 'crdUtl', 'pastDue_to_cl', 'pastDue_to_hc', 'pastDue_to_blnc', 'monthly_payment_amt', 'loan_duration', 'months_until_maturity', 'acctAge_to_loanDuration', 'acctAge_to_loanDuration_2', 'pastDue_to_monthlyPmt', 'is_transfer', 'is_member', 'is_deferred', 'is_settled', 'is_active', 'is_opened_in_last_12m', 'is_bnkr', 'is_not_bnkr', 'is_bnkr_not_discharged', 'is_bnkr_discharged', 'is_bnkr_ch7', 'is_bnkr_ch13', 'is_closed', 'is_open', 'is_open_active', 'is_dq30', 'is_dq60', 'is_dq90', 'is_dq120_excluding_CO_codes', 'is_dq120+', 'is_derog', 'is_dq', 'is_non_derog', 'has_recent_payment', 'is_never_dq', 'is_dq30+_in_last_24_months', 'is_dq60+_in_last_24_months', 'is_dq90+_in_last_24_months', 'is_dq120+_in_last_24_months', 'is_CO_in_last_24_months', 'is_dq30+_in_last_12_months', 'is_dq60+_in

In [111]:
preprocess_dict['PaymentPatternsAggregatorV2'].__dict__.keys()

dict_keys(['_init_args', '_init_kwargs', 'transformer_name', 'payment_patterns', 'patterns', 'rate', 'trim', 'keep', 'placeholder', 'report_date', 'month_ranges', 'name', 'zest_ppt_name', 'ppt_len_name', 'return_zest_ppt', 'return_ppt_len', '_rate', '_new_features', 'date_delta_col', '_other_params', 'input_dtypes', 'output_dtypes'])

In [112]:
payment_df_test = preprocess_dict['PaymentPatternsAggregatorV2'].transform(preprocess_dict['months_since_rptDate'].transform(mapped_df))

In [113]:
payment_features = preprocess_dict['PaymentPatternsAggregatorV2']._new_features

In [114]:
df_test = payment_df_test[payment_features]

In [115]:
df_ours = data_for_payment_columns[payment_features]

In [116]:
payment_df_test.columns[payment_df_test.columns.duplicated()]

Index([], dtype='object')

In [117]:
len(df_test.columns)

88

In [118]:
df_test

,number_DQ30+_3_months,percent_DQ30+_3_months,number_DQ30+_6_months,percent_DQ30+_6_months,number_DQ30+_12_months,percent_DQ30+_12_months,number_DQ30+_24_months,percent_DQ30+_24_months,number_DQ30+_48_months,percent_DQ30+_48_months,...,percent_DQ90_3_months,number_DQ90_6_months,percent_DQ90_6_months,number_DQ90_12_months,percent_DQ90_12_months,number_DQ90_24_months,percent_DQ90_24_months,number_DQ90_48_months,percent_DQ90_48_months,months_since_most_recent_DQ90
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,1.0,0.5,1.0,0.2,1.0,0.090909,1.0,0.043478,1.0,0.021277,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,1.0,0.5,1.0,0.2,1.0,0.090909,1.0,0.043478,1.0,0.021277,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9718,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN
9719,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN
9720,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN
9721,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN


In [119]:
df_ours

,number_DQ30+_3_months,percent_DQ30+_3_months,number_DQ30+_6_months,percent_DQ30+_6_months,number_DQ30+_12_months,percent_DQ30+_12_months,number_DQ30+_24_months,percent_DQ30+_24_months,number_DQ30+_48_months,percent_DQ30+_48_months,...,percent_DQ90_3_months,number_DQ90_6_months,percent_DQ90_6_months,number_DQ90_12_months,percent_DQ90_12_months,number_DQ90_24_months,percent_DQ90_24_months,number_DQ90_48_months,percent_DQ90_48_months,months_since_most_recent_DQ90
0,0,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaN
1,0,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaN
2,1,0.5,1,0.2,1,0.090909,1,0.043478,1,0.021277,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaN
3,0,0.0,0,0.0,0,0.000000,0,0.000000,0,0.000000,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaN
4,1,0.5,1,0.2,1,0.090909,1,0.043478,1,0.021277,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9718,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,NaN
9719,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,NaN
9720,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,NaN
9721,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,NaN


In [120]:
np.unique((df_test == df_ours) | (df_test.isna() & df_ours.isna()))

array([ True])



## Coalescing and Dynamic Placeholders

After payment history, several cleanup steps run. From the [preprocess asset](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/equifax/cms_6/fe2/trade.json):

**CoalesceV2** - if `high_credit_amt` is NA, fill with `chargeoff_amt`:
```python
{'type': 'CoalesceV2',
 'params': {'feature_1': 'high_credit_amt', 'feature_2': 'chargeoff_amt',
  'new_feature': 'high_credit_amt', 'mode': 'na', 'input_dtype': 'numeric'}}
```
The borrower must have reached at least that chargeoff amount in debt or the charge off could not have happened.

**DynamicPlaceholderV2** - set `termDur` to null for revolving/open/charge card accounts:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'termDur', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'PORTFOLIO_TYPE', 'op': 'in', 'value': ['R', 'O', 'C']},
    {'column': 'accountType', 'op': 'in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'any'},
  'raw_feature_input_dtype': 'numeric'}}
```
Loan duration only makes sense for installment loans with fixed repayment schedules.

**DynamicPlaceholderV2** - set `credit_limit` to null for non-revolving accounts:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'credit_limit', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'PORTFOLIO_TYPE', 'op': 'not_in', 'value': ['R', 'O', 'C']},
    {'column': 'accountType', 'op': 'not_in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'all'},
  'raw_feature_input_dtype': 'numeric'}}
```
Credit limit only makes sense for revolving accounts.

**DynamicPlaceholderV2** - set `credit_limit` to null when it's 0 for credit card accounts (likely bad data):
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'credit_limit', 'new_value': None,
  'raw_feature_input_dtype': 'numeric',
  'filter_params': {'conditions': [
    {'column': 'credit_limit', 'op': '==', 'value': 0},
    {'column': 'accountType', 'op': 'in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'all'}}}
```

## Bivariate Ratios

A series of `BivariateComputeV2` operations compute ratios. For example:
```python
{'type': 'BivariateComputeV2',
 'params': {'feature_1': 'balance_amt', 'feature_2': 'credit_limit',
  'op': 'divide', 'new_feature': 'crdUtl'}}
```

The full set:
- `blnc_to_hc` = balance / high_credit - how current debt compares to max exposure
- `hc_to_cl` = high_credit / credit_limit - max historical utilization
- `crdUtl` = balance / credit_limit - current utilization
- `pastDue_to_cl` = past due / credit_limit - severity of delinquency relative to trust extended
- `pastDue_to_hc` = past due / high_credit - severity relative to max exposure
- `pastDue_to_blnc` = past due / balance - severity relative to current debt
- `monthly_payment_amt` = scheduled_payment * termFreqMult - monthly payment amount
- `loan_duration` = termDur * termFreqMult - total loan duration in months
- `months_until_maturity` = loan_duration - months_since_openDate - how many months left
- `acctAge_to_loanDuration` = months_since_openDate / loan_duration - how far along the loan is
- `pastDue_to_monthlyPmt` = past due / monthly payment - how many months of payments are overdue

For `acctAge_to_loanDuration`, values > 1 with positive balance are set to null:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'acctAge_to_loanDuration', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'acctAge_to_loanDuration', 'op': '>', 'value': 1},
    {'column': 'balance_amt', 'op': '>=', 'value': 0}],
   'outer_op': 'all'},
  'raw_feature_input_dtype': 'numeric'},
 'notes': "for open_installment loans, this could be bad info or a sign of late pmts..."}
```

If the loan is past its duration but still has a balance, it could be bad data or a sign of late payments. Since we want to monotonically constrain this feature (more age = less risky), we null out these cases so they don't get treated as low risk.

## Filter Flags (Boolean Indicators)

The bulk of preprocessing creates `FilterV2` boolean columns (1 or 0) that the AggregationEngine uses to define which tradelines to group over. For example:
```python
{'type': 'FilterV2',
 'params': {'new_feature': 'is_never_dq',
  'conditions': [
    {'column': 'dqDate', 'op': 'is_na', 'value': None},
    {'column': 'number_DQ30+_48_months', 'op': '==', 'value': 0},
    {'column': 'is_non_derog', 'op': '==', 'value': 1},
    {'column': 'is_deferred', 'op': 'not_in', 'value': [1]},
    {'column': 'RATE_STATUS_CODE', 'op': 'not_in',
     'value': ['2','6','8','3','9','4','G','M','#','7','Z','5']}],
  'outer_op': 'all'}}
```

This checks ALL conditions: no historic dqDate, zero DQ30+ in payment history, currently non-derogatory, not deferred, and rate status is clean. Filters chain together - `is_non_derog` is itself a filter defined earlier that checks `RATE_STATUS_CODE in ['1','2']` AND `is_deferred != 1` AND `is_derog != 1`.

The filters fall into these categories:

**Account status**: `is_open`, `is_closed`, `is_open_active`, `is_transfer`, `is_deferred`, `is_settled`, `is_member`, `has_recent_payment`

**Delinquency status**: `is_dq30`, `is_dq60`, `is_dq90`, `is_dq120+`, `is_derog`, `is_dq`, `is_non_derog`, `is_never_dq`, plus time-windowed versions like `is_dq30+_in_last_24_months`, `is_CO_in_last_12_months`, etc.

**Bankruptcy**: `is_bnkr`, `is_bnkr_ch7`, `is_bnkr_ch13`, `is_bnkr_discharged`

**Account type**: `is_auto`, `is_personal`, `is_credit_card`, `is_charge_card`, `is_homeequity`, `is_mortgage`, `is_first_mortgage`, `is_education`, `is_businessloan`, etc.

**Portfolio type**: `is_installment`, `is_revolving`, `is_loc`, `is_open_ended`, `is_mortgage`

**Utilization buckets**: `has_utilization_over_25_percent`, `has_utilization_over_50_percent`, `has_utilization_over_100_percent` (same for historic utilization using `hc_to_cl`)

**Ownership**: `is_individual`, `is_joint`, `is_joint_not_mortgage`

**Negative events**: `is_chargeoff`, `is_negative`, `is_closed_negative`, `is_foreclosure`, `is_reposession_surrender`, `is_collections`

## Row Dropping

At the end, three `RowDropperV2` operations remove tradelines where the person is not financially responsible:
```python
{'type': 'RowDropperV2',
 'params': {'transformer_name': 'drop_authorized',
  'conditions': [{'column': 'ecoa', 'op': 'in', 'value': ['A']}],
  'outer_op': 'any'},
 'notes': 'dropping authorized trades'}
```

- `ecoa == 'A'` (authorized user - not their account)
- `ecoa == 'T'` (terminated - no longer associated)
- `ecoa == 'X'` (deceased)

In [121]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

OrderedDict([('Mappers',
              <model_engine.feature_engine_V2.listed_objects_engines.MapperV2 at 0x7fb1d494f070>),
             ('Preprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fb1d494f640>),
             ('Postprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fb0dc17ca30>)])

In [122]:
preprocessed_df = node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Preprocessors'].transform(mapped_df)

In [123]:
preprocessed_df

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,has_utilization_over_25_percent,has_utilization_over_50_percent,has_utilization_over_100_percent,has_historic_utilization_over_25_percent,has_historic_utilization_over_50_percent,has_historic_utilization_over_100_percent,has_balance_greater_than_hc,is_individual,is_joint,is_joint_not_mortgage
0,303683_1_276_12,2023Q1,2023-04-21,1,FULL-Header,1,PT,None,DC,03202023,...,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,303683_1_276_12,2023Q1,2023-04-21,2,FULL-Header,2,PT,None,DC,03202023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,303683_1_276_12,2023Q1,2023-04-21,3,FULL-Header,3,PT,None,FY,03032023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,303683_1_276_12,2023Q1,2023-04-21,4,FULL-Header,4,PT,None,FA,02282023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,303683_1_276_12,2023Q1,2023-04-21,5,FULL-Header,5,PT,None,ON,03032023,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9718,305378_2_276_12,2023Q1,2023-04-29,22,FULL-Header,22,PT,None,FF,09182016,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0
9719,305378_2_276_12,2023Q1,2023-04-29,23,FULL-Header,23,PT,None,BB,06282016,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9720,305378_2_276_12,2023Q1,2023-04-29,24,FULL-Header,24,PT,None,FM,05072015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9721,305378_2_276_12,2023Q1,2023-04-29,25,FULL-Header,25,PT,None,BB,08132014,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


# Row Dropper

# Aggregation Engine

In [124]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

OrderedDict([('Mappers',
              <model_engine.feature_engine_V2.listed_objects_engines.MapperV2 at 0x7fb1d494f070>),
             ('Preprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fb1d494f640>),
             ('Postprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fb0dc17ca30>)])

In [125]:
post_processor = node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Postprocessors']

In [126]:
normalized_df = post_processor.transform(preprocessed_df)

In [127]:
normalizer_location ='s3://power-client-data-staging/NORMALIZED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/'

In [128]:
trade_data = data_loader(asset['data']['trade']['data'], asset={'info': {'key': 'ZEST_KEY'}}).reset_index()[trade_columns]

In [129]:

bucket_name = "power-client-data-staging"
s3 = handler.S3Handler(bucket_name=bucket_name)


In [130]:
base_normalized_path = 'NORMALIZED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1'

In [131]:
normalized_locations = s3.list(base_normalized_path)

In [132]:
import pandas as pd

dfs = []

for sub_path in normalized_locations:
    full_path = f"s3://power-client-data-staging/{base_normalized_path}/{sub_path}"
    
    df = pd.read_parquet(full_path)
    dfs.append(df)

full_normalized_df = pd.concat(dfs, ignore_index=True)

In [133]:
full_normalized_df.columns

Index(['ZEST_KEY', 'date_of_request', 'balance_amt', 'credit_limit',
       'high_credit_amt', 'pastDueAmt', 'openDate', 'closedDate', 'rptDate',
       'lstPmtDate',
       ...
       'has_utilization_over_25_percent', 'has_utilization_over_50_percent',
       'has_utilization_over_100_percent',
       'has_historic_utilization_over_25_percent',
       'has_historic_utilization_over_50_percent',
       'has_historic_utilization_over_100_percent',
       'has_balance_greater_than_hc', 'is_individual', 'is_joint',
       'is_joint_not_mortgage'],
      dtype='object', length=192)

In [134]:
[feature for feature in normalized_df.columns if feature  in full_normalized_df.columns]

['ZEST_KEY',
 'date_of_request',
 'balance_amt',
 'credit_limit',
 'high_credit_amt',
 'pastDueAmt',
 'openDate',
 'closedDate',
 'rptDate',
 'lstPmtDate',
 'dqDate',
 'accountType',
 'portfolioType',
 'ecoa',
 'chargeoff_amt',
 'constant_placeholder',
 'months_since_rptDate',
 'months_since_openDate',
 'months_since_lstPmtDate',
 'number_DQ30+_3_months',
 'percent_DQ30+_3_months',
 'number_DQ30+_6_months',
 'percent_DQ30+_6_months',
 'number_DQ30+_12_months',
 'percent_DQ30+_12_months',
 'number_DQ30+_24_months',
 'percent_DQ30+_24_months',
 'number_DQ30+_48_months',
 'percent_DQ30+_48_months',
 'months_since_most_recent_DQ30+',
 'number_DQ60+_3_months',
 'percent_DQ60+_3_months',
 'number_DQ60+_6_months',
 'percent_DQ60+_6_months',
 'number_DQ60+_12_months',
 'percent_DQ60+_12_months',
 'number_DQ60+_24_months',
 'percent_DQ60+_24_months',
 'number_DQ60+_48_months',
 'percent_DQ60+_48_months',
 'months_since_most_recent_DQ60+',
 'number_DQ90+_3_months',
 'percent_DQ90+_3_months',
 'n

In [135]:
[feature for feature in normalized_df.columns if feature not in full_normalized_df.columns]

['quarter',
 'appDate',
 'SEG_SEQ',
 'SEG_PARENT',
 'SEG_PARENT_SEQ',
 'SEGMENT_TYPE',
 'CUSTOMER_NAME',
 'DATE_REPORTED',
 'DATE_OPENED',
 'HIGH_CREDIT',
 'CREDIT_LIMIT',
 'BALANCE',
 'PAST_DUE_AMOUNT',
 'AUTOMATED_UPDATE_INDICATOR',
 'MONTHS_REVIEWED',
 'ECOA_DESIGNATOR',
 'ACCOUNT_NUMBER',
 '30_DAY_COUNTERS',
 '60_DAY_COUNTERS',
 '90+_DAY_COUNTERS',
 'PREVIOUS_HIGH_RATE_1',
 'PREVIOUS_HIGH_DATE_1',
 'PREVIOUS_HIGH_RATE_2',
 'PREVIOUS_HIGH_DATE_2',
 'PREVIOUS_HIGH_RATE_3',
 'PREVIOUS_HIGH_DATE_3',
 'DFD_DLA',
 'NARRATIVE_CODE_3',
 'NARRATIVE_CODE_4',
 'ACCOUNT_TYPE',
 'LAST_PAYMENT_DATE',
 'CLOSED_DATE',
 'DMD_REPORTED',
 'ACTUAL_PAYMENT_AMOUNT',
 'SCHEDULED_PAYMENT_AMOUNT',
 'TERMS_FREQUENCY',
 'TERMS_DURATION',
 'INDICATOR',
 'NAME',
 'CREDITOR_CLASSIFICATION',
 'ORIGINAL_CHARGE_OFF_AMOUNT',
 'DEFERRED_PAYMENT_START_DATE',
 'BALLOON_PAYMENT_AMOUNT',
 'BALLOON_PAYMENT_DUE_DATE',
 'MORTGAGE_ID_NUMBER',
 'PREVIOUS_HIGH_RATE_BEFORE_HISTORY',
 'PREVIOUS_HIGH_DATE_BEFORE_HISTORY',
 'NARR

# Aggregation Engine

In [137]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset.keys()

dict_keys(['aggregator', 'postprocess', 'table_name', 'feature_engineering', 'fe_version', 'keep_features'])

In [138]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator'].keys()

dict_keys(['key', 'table', 'features', 'feature_groups', 'secondary_filter_groups', 'aggregations', 'missing_map', 'key_factor_schemes', 'keep_features'])

In [153]:
[feature for feature in trade_columns if 'never' in feature.lower()]

['trade_max_credit_limit__never_delinquent_revolving',
 'trade_months_since_oldest_account_opened__never_delinquent_open_revolving',
 'trade_months_since_last_payment__never_delinquent_open_revolving',
 'trade_sum_loan_amount__never_delinquent_installment',
 'trade_months_since_most_recently_opened__never_delinquent_open_cc',
 'trade_count__never_delinquent_accounts',
 'trade_max_utilization__never_delinquent_cc',
 'trade_mean_acctAge_to_loanDuration__never_delinquent_open_installment',
 'trade_months_since_last_payment__never_delinquent_installment',
 'trade_mean_loan_amount__never_delinquent_closed_installment',
 'trade_percent_never_dq__mortgage',
 'trade_mean_credit_limit__never_delinquent_cc',
 'trade_mean_credit_limit__never_delinquent_open_cc',
 'trade_mean_loan_amount__never_delinquent_closed_personal',
 'trade_count__never_delinquent_mortgage_accounts',
 'trade_sum_balance__never_delinquent_open_accounts',
 'trade_mean_account_age__never_delinquent_open_accounts',
 'trade_max_

In [139]:
type(node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features'])

dict

# For each dictionary in feature, it tells how how we are going to aggregate each feature

In [140]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features'].keys()

dict_keys(['count', 'months_since_most_recently_opened', 'months_since_oldest_account_opened', 'mean_account_age', 'balance', 'amount_past_due', 'monthly_payment_amount', 'balance_to_hc', 'percent_unpaid', 'historic_utilization', 'utilization', 'past_due_to_limit', 'past_due_to_high_balance', 'past_due_to_balance', 'past_due_to_monthly_payment', 'credit_limit', 'loan_amount', 'high_balance', 'acctAge_to_loanDuration', 'months_since_most_recent_DQ30_or_greater', 'months_since_most_recent_DQ60_or_greater', 'months_since_most_recent_DQ90_or_greater', 'months_since_most_recent_DQ120_or_greater', 'number_of_DQ30_or_greater_in_last_6_months', 'number_of_DQ30_in_last_6_months', 'number_of_DQ30_in_last_12_months', 'number_of_DQ30_in_last_24_months', 'number_of_DQ60_in_last_6_months', 'number_of_DQ60_in_last_12_months', 'number_of_DQ60_in_last_24_months', 'number_of_DQ90_in_last_6_months', 'number_of_DQ90_in_last_12_months', 'number_of_DQ90_in_last_24_months', 'number_of_DQ120_or_greater_in_las

In [152]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].transformers

OrderedDict([('Aggregator',
              <feature_engine_parts.fe_parts_V2.aggregators.aggregator.FilterAggregationV2 at 0x7fb0dc17caf0>)])

In [141]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features']['months_since_most_recently_opened']

{'feature': 'months_since_openDate',
 'agg': 'min',
 'definition': '<agg_definition> months since open date across <group>',
 'key_factor_mapping': 'Length of history on <group>'}

In [142]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['feature_groups'].keys()

dict_keys(['all', 'all_with_subgroups', 'count_only', 'bankruptcy', 'basic', 'open_accounts', 'closed_accounts', 'open_revolving', 'closed_revolving', 'all_revolving', 'open_installment', 'closed_installment', 'all_installment', 'open_charge_card', 'closed_charge_card', 'all_charge_card', 'time_based', 'months_since_dq_features', 'dq_frequency_features_6m', 'dq_frequency_features_12m', 'dq_frequency_features_24m', 'percent_with_past_dq', 'utilization_features', 'historic_utilization_features', 'derog_features', 'non_derog_features', 'past_due_features', 'negative_features', 'positive_features', 'last_payment_date_features', 'diversity_features'])

In [146]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['feature_groups']['all']

['count',
 'diversity_of_accounts',
 'diversity_of_ecoa',
 'months_since_oldest_account_opened',
 'months_since_last_payment',
 'months_since_last_activity',
 'percent_recently_open',
 'percent_open',
 'percent_never_dq',
 'percent_non_derog',
 'percent_derog',
 'percent_DQ120_or_greater',
 'months_since_most_recent_DQ30_or_greater',
 'months_since_most_recent_DQ60_or_greater',
 'months_since_most_recent_DQ90_or_greater',
 'months_since_most_recent_DQ120_or_greater',
 'percent_accounts_with_DQ30_or_greater_in_last_24_months',
 'percent_accounts_with_DQ60_or_greater_in_last_24_months',
 'percent_accounts_with_DQ90_or_greater_in_last_24_months',
 'percent_accounts_with_DQ120_or_greater_in_last_24_months',
 'percent_accounts_with_DQ30_or_greater_in_last_12_months',
 'percent_accounts_with_DQ60_or_greater_in_last_12_months',
 'percent_accounts_with_DQ90_or_greater_in_last_12_months',
 'percent_accounts_with_DQ120_or_greater_in_last_12_months',
 'percent_accounts_with_DQ30_or_greater_in_las

In [149]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features']['months_since_last_activity']

{'feature': 'months_since_rptDate',
 'agg': 'min',
 'definition': '<agg_definition> months since report date across <group>',
 'key_factor_mapping': 'Time since activity on <group>'}

In [147]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['feature_groups']['closed_installment']

['months_since_last_activity',
 {'feature': 'balance', 'aggs': ['sum', 'mean', 'min', 'max']},
 {'feature': 'percent_unpaid', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'loan_amount', 'aggs': ['mean', 'max', 'sum']},
 {'feature': 'amount_past_due', 'aggs': ['sum', 'mean', 'max', 'min']},
 {'feature': 'past_due_to_balance', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'past_due_to_high_balance', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'past_due_to_monthly_payment', 'aggs': ['mean', 'max', 'min']}]

In [143]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['feature_groups']['negative_features']

[{'feature': 'amount_past_due', 'aggs': ['sum', 'mean', 'max', 'min']},
 {'feature': 'past_due_to_balance', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'past_due_to_high_balance', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'past_due_to_limit', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'percent_unpaid', 'aggs': ['mean', 'max', 'min']},
 {'feature': 'past_due_to_monthly_payment', 'aggs': ['mean', 'max', 'min']}]

## The feature groups groups these features together

In [144]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features']['amount_past_due']

{'feature': 'pastDueAmt',
 'agg': ['sum', 'mean', 'min', 'max'],
 'definition': '<agg_definition> amount past due across <group>',
 'key_factor_mapping': 'Amount past due on <group>'}

In [145]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['primary_filters']

KeyError: 'primary_filters'

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['secondary_filter_groups'].keys()

# The secondary filter groups tells us subgroups along with sets of features and aggregations we will apply to those subgroups to

# We filter on where is_derog is true

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['secondary_filter_groups'].keys()

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['secondary_filter_groups']['derog']

In [ ]:
exclusions_derog = node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['secondary_filter_groups']['derog']['exclusions']

In [ ]:
[feature for feature in node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['feature_groups'].keys() if feature not in exclusions_derog]

# We can see for negative we won't do aggregations for non_derog_Features, derog_features, percent_with_past_dq

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['features']['percent_derog']

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator'].keys()

In [ ]:
node_dict['trade FE'].transformer.transformers['AggregationEngine'].asset['aggregator']['aggregations'].keys()

In [ ]:
[feature for feature in trade_data.columns if 'derog' in feature and 'non' not in feature]

In [ ]:
trade_data.columns

# Look at [this](https://zestfinance.atlassian.net/wiki/spaces/DS/pages/2789933081/Anatomy+of+an+AggregationEngine+Asset)